In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

c:\Users\sneha\Desktop\LANGCHAIN_TUTORIAL\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 1(A): Indexing (Document Ingestion)

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi


In [ ]:
import sys
!{sys.executable} -m pip install --upgrade youtube-transcript-api


In [ ]:
video_id = "kjBOesZCoqc" # only the ID, not full URL

try:
    # Create API object
    ytt_api = YouTubeTranscriptApi()

    # Fetch transcript (English)
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # Convert to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Hey everyone! So I'm pretty excited about the next sequence of videos that I'm doing. They'll be about linear algebra, which, as a lot of you know,  is one of those subjects that's required knowledge for just about any technical  discipline, but it's also, I've noticed, generally poorly understood by students  taking it for the first time. A student might go through a class and learn how to compute lots of things,  like matrix multiplication, or the determinant, or cross products,  which use the determinant, or eigenvalues, but they might come out without really  understanding why matrix multiplication is defined the way that it is,  why the cross product has anything to do with the determinant,  or what an eigenvalue really represents. Oftentimes, students end up well practiced in the numerical operations of matrices,  but are only vaguely aware of the geometric intuitions underlying it all. But there's a fundamental difference between understanding linear  algebra on a numeric level 

In [ ]:
for i in transcript_list:
    print(i)

FetchedTranscriptSnippet(text='Hey everyone!', start=11.979, duration=1.001)
FetchedTranscriptSnippet(text="So I'm pretty excited about the next sequence of videos that I'm doing.", start=13.36, duration=2.9)
FetchedTranscriptSnippet(text="They'll be about linear algebra, which, as a lot of you know, ", start=16.74, duration=3.124)
FetchedTranscriptSnippet(text="is one of those subjects that's required knowledge for just about any technical ", start=19.864, duration=4.032)
FetchedTranscriptSnippet(text="discipline, but it's also, I've noticed, generally poorly understood by students ", start=23.896, duration=4.082)
FetchedTranscriptSnippet(text='taking it for the first time.', start=27.978, duration=1.462)
FetchedTranscriptSnippet(text='A student might go through a class and learn how to compute lots of things, ', start=30.1, duration=4.19)
FetchedTranscriptSnippet(text='like matrix multiplication, or the determinant, or cross products, ', start=34.29, duration=3.694)
FetchedTranscript

Step 1(B): Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:

len(chunks)

7

In [ ]:
chunks[2]

Document(metadata={}, page_content="statistics, economics, or even math itself. Once you're in a class, or a job for that matter,  that assumes fluency with linear algebra, the way that your  professors or your co-workers apply that field could seem like utter magic. They'll very quickly know what the right tool to use is and what the answer  roughly looks like in a way that would seem like computational wizardry if  you assume that they're actually crunching all the numbers in their head. Here, as an analogy, imagine that when you first learned about the  sine function in trigonometry, you were shown this infinite polynomial. This, by the way, is how your calculator evaluates the sine function. For homework, you might be asked to practice computing approximations of the sine  function by plugging in various numbers to the formula and cutting it off at a reasonable  point. And in fairness, let's say you had a vague idea that this was  supposed to be related to triangles, but exactly ho


Step 1(C) and 1(D): Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:

vector_store.index_to_docstore_id

{0: '8877d03b-195a-4643-a2bf-55cf54348117',
 1: '8a5c46ae-6408-4001-86c3-2aebb8d972ea',
 2: '13f0d5e6-6458-43c5-8055-32408a6565e3',
 3: '311998a1-3868-4ebd-8d74-4b2f8975892c',
 4: '25d9dfe5-f8f7-496b-931c-d2e899079d11',
 5: '5d4ac7f1-0778-4bab-ad33-47d169f14736',
 6: '2768d292-f9b7-4378-a81e-2a04353493bc'}

In [ ]:
vector_store.get_by_ids(['5d4ac7f1-0778-4bab-ad33-47d169f14736'])

[Document(id='5d4ac7f1-0778-4bab-ad33-47d169f14736', metadata={}, page_content="So this brings me to the upcoming videos. The goal is to create a short, binge-watchable series animating those intuitions from the  basics of vectors up through the core topics that make up the essence of linear algebra. I'll put out one video per day for the next five days,  then after that put out a new chapter every one to two weeks. I think it should go without saying that you cannot learn a full  subject with a short series of videos, and that's just not the goal here. But what you can do, especially with this subject,  is lay down all the right intuitions so the learning that you do moving forward is as  productive and fruitful as it can be. I also hope this can be a resource for educators who are  teaching courses that assume fluency with linear algebra,  giving them a place to direct students that need a quick brush up. I'll do what I can to keep things well paced throughout,  but it's hard to simu


Step 2: Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:

retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A1C64278F0>, search_kwargs={'k': 5})

In [ ]:

retriever.invoke('linear regression ')

[Document(id='13f0d5e6-6458-43c5-8055-32408a6565e3', metadata={}, page_content="statistics, economics, or even math itself. Once you're in a class, or a job for that matter,  that assumes fluency with linear algebra, the way that your  professors or your co-workers apply that field could seem like utter magic. They'll very quickly know what the right tool to use is and what the answer  roughly looks like in a way that would seem like computational wizardry if  you assume that they're actually crunching all the numbers in their head. Here, as an analogy, imagine that when you first learned about the  sine function in trigonometry, you were shown this infinite polynomial. This, by the way, is how your calculator evaluates the sine function. For homework, you might be asked to practice computing approximations of the sine  function by plugging in various numbers to the formula and cutting it off at a reasonable  point. And in fairness, let's say you had a vague idea that this was  suppose

Step 3: Augumentation

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.

    Context:
    {context}

    Question:
    {question}
    """,
    input_variables=["context", "question"]
)


In [ ]:
question = "what is linear algebra  ?"
retrieved_docs = retriever.invoke(question)

In [ ]:

retrieved_docs

[Document(id='8877d03b-195a-4643-a2bf-55cf54348117', metadata={}, page_content="Hey everyone! So I'm pretty excited about the next sequence of videos that I'm doing. They'll be about linear algebra, which, as a lot of you know,  is one of those subjects that's required knowledge for just about any technical  discipline, but it's also, I've noticed, generally poorly understood by students  taking it for the first time. A student might go through a class and learn how to compute lots of things,  like matrix multiplication, or the determinant, or cross products,  which use the determinant, or eigenvalues, but they might come out without really  understanding why matrix multiplication is defined the way that it is,  why the cross product has anything to do with the determinant,  or what an eigenvalue really represents. Oftentimes, students end up well practiced in the numerical operations of matrices,  but are only vaguely aware of the geometric intuitions underlying it all. But there's a 

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"Hey everyone! So I'm pretty excited about the next sequence of videos that I'm doing. They'll be about linear algebra, which, as a lot of you know,  is one of those subjects that's required knowledge for just about any technical  discipline, but it's also, I've noticed, generally poorly understood by students  taking it for the first time. A student might go through a class and learn how to compute lots of things,  like matrix multiplication, or the determinant, or cross products,  which use the determinant, or eigenvalues, but they might come out without really  understanding why matrix multiplication is defined the way that it is,  why the cross product has anything to do with the determinant,  or what an eigenvalue really represents. Oftentimes, students end up well practiced in the numerical operations of matrices,  but are only vaguely aware of the geometric intuitions underlying it all. But there's a fundamental difference between understanding linear  algebra on a numeric level

In [ ]:

final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n\n    Context:\n    Hey everyone! So I'm pretty excited about the next sequence of videos that I'm doing. They'll be about linear algebra, which, as a lot of you know,  is one of those subjects that's required knowledge for just about any technical  discipline, but it's also, I've noticed, generally poorly understood by students  taking it for the first time. A student might go through a class and learn how to compute lots of things,  like matrix multiplication, or the determinant, or cross products,  which use the determinant, or eigenvalues, but they might come out without really  understanding why matrix multiplication is defined the way that it is,  why the cross product has anything to do with the determinant,  or what an eigenvalue really represents. Oftentimes, students end up well practiced in the numerical operations of matrices,  but are only vaguely aware of the

Step 4: Generation


In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

The provided transcript does not explicitly define what linear algebra is. It mentions that it is a subject required for almost any technical discipline and discusses different levels of understanding it (numeric vs. geometric).


Building Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:

def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("Algebra is scalar or vector ?")


{'context': "Hey everyone! So I'm pretty excited about the next sequence of videos that I'm doing. They'll be about linear algebra, which, as a lot of you know,  is one of those subjects that's required knowledge for just about any technical  discipline, but it's also, I've noticed, generally poorly understood by students  taking it for the first time. A student might go through a class and learn how to compute lots of things,  like matrix multiplication, or the determinant, or cross products,  which use the determinant, or eigenvalues, but they might come out without really  understanding why matrix multiplication is defined the way that it is,  why the cross product has anything to do with the determinant,  or what an eigenvalue really represents. Oftentimes, students end up well practiced in the numerical operations of matrices,  but are only vaguely aware of the geometric intuitions underlying it all. But there's a fundamental difference between understanding linear  algebra on a n

In [ ]:

parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [46]:
main_chain.invoke('Can you summarize the video')

'The upcoming videos will be about linear algebra, a subject often poorly understood by students who learn computations without grasping the underlying geometric intuitions. The speaker notes that while students might learn how to compute things like matrix multiplication or determinants, they often don\'t understand *why* these operations are defined as they are or what they represent visually.\n\nThe goal of this new "short, binge-watchable series" is to animate these visual intuitions, starting from the basics of vectors and covering core linear algebra topics. The series aims to clarify the connection between the computations and their geometric meaning, making the subject more reasonable and practical, especially since computers typically handle numerical calculations while humans focus on conceptual understanding.\n\nThe series will release one video per day for five days, followed by a new chapter every one to two weeks. It\'s not intended to teach the full subject, but rather t